# 4. Model training and evaluation

Load the reviewed training, validation and test tables exported by [the preprocessing notebook](Crop_Four_Month_Windows.ipynb). Train models from these saved inputs and preserve their evaluation results.

## Before running

1. Run preprocessing through **Step 14: Export the prepared training data**.
2. Keep `training_handoff.json`, `partitioned_features.csv`, `configuration.json` and `coverage.json` together.
3. Select `.venv` as the kernel and run this notebook from the repository folder.
4. Install `requirements-notebooks.txt` if needed.

This notebook starts in a **fresh kernel**. It reads prepared files directly and does not require imagery downloads, Earth Engine authentication or variables from the other notebook.

**Current design:** external-only, local-only and combined Random Forests, each with 40-band or 64-band-plus-index features. This split preserves the existing experiment; the thesis-methodology differences identified earlier have not been implemented.

**Reading outputs:** each executable cell has an adjacent Markdown output guide. Read `Output label: <number>` as the label plus the value from your current run; the guide explains the counting unit, purpose and interpretation. A zero count can mean that a stage has no eligible inputs, rather than that the code failed.


## 1. Set the training-input path

The default path follows `report_root` in `settings.local.json`. Change `handoff_path` below to use a bundle from another preprocessing run or PC. Its companion files are located relative to the manifest, making the bundle portable.

Outputs are written to `models/` and `training_package/` beside the `training_input/` folder. Update `training_output_folder` if you want to preserve multiple training runs.

**Estimated-window experiment:** set `handoff_path` to `exploratory_training_input/training_handoff.json`. The saved policy is checked before fitting. Results from this bundle go to `exploratory_estimated/` and remain separate from primary models.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Training handoff: <path>` | The saved-data manifest this notebook will load. Confirm whether you chose the primary or exploratory input bundle. |
| `Output folder: <path>` | The preliminary destination derived from the handoff location. The next cell moves exploratory results into a separate exploratory_estimated folder after reading the policy. |
<!-- output-guide:end -->


In [ ]:
from pathlib import Path
import json
import season_workflow

settings_path = Path("settings.local.json")
local_settings = (
    json.loads(settings_path.read_text(encoding="utf-8")) if settings_path.exists() else {}
)
report_folder = (
    Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
)
season_folder = report_folder.parent / "analysis" / "pilot_3000" / "seasons"
handoff_path = season_folder / "training_input" / "training_handoff.json"
training_output_folder = handoff_path.parent.parent
print("Training handoff:", handoff_path)
print("Output folder:", training_output_folder)


## 2. Load and check the prepared inputs

Verify the feature schema, row count and hashes before loading. An edited or incomplete bundle requires a fresh preprocessing export.

The table includes crop labels, window dates, evidence, field IDs and partitions. Only satellite feature columns become model inputs. Validation rows remain reserved; independent test rows are used only for evaluation. Field identity and complete-feature checks run again before fitting.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Preparation status: prepared or pending_inputs` | The export manifest status: prepared means it recorded rows with a frozen configuration; pending_inputs means rows/settings were not ready at export. It is not a measured accuracy result or a guarantee of all training prerequisites. |
| `Prepared pixel windows: <number>` | Rows loaded from the partitioned table, including train, validation and test assignments. Zero means this bundle cannot fit models. |
| `Configured crops: [<class>, ...]` | The class scheme expected in each training source and the common test set. Training stops if a required class is missing. |
| `Field/sample counts: [<records>]` | Pixel counts and distinct field IDs grouped by source, partition, crop, label status and timing status. Fields can appear in several reporting groups; do not sum subgroup field counts as a unique overall total. |
| `Window policy: reference_supported_only or exploratory_external_estimates` | Controls whether only reference-supported windows or also qualified external training estimates are allowed. Estimated local/test truth is prohibited. |
| `Window statuses: {<status>: <number>}` | Input-row counts by window evidence category, not independent fields. This shows whether estimates are present in the loaded bundle. |
| `Model output folder: <path>` | The actual destination after the input policy is read. Exploratory models are isolated from primary outputs. |
<!-- output-guide:end -->


In [ ]:
partitioned_rows, configuration, preprocessing_coverage, handoff_manifest = (
    season_workflow.load_training_handoff(handoff_path)
)
print("Preparation status:", handoff_manifest["status"])
print("Prepared pixel windows:", len(partitioned_rows))
print("Configured crops:", configuration["study_classes"])
print("Field/sample counts:", season_workflow.sample_counts(partitioned_rows))
# Keep exploratory models separate even when only the input path was changed.
window_policy = configuration.get("window_policy", "reference_supported_only")
training_output_folder = handoff_path.parent.parent
if window_policy == "exploratory_external_estimates":
    training_output_folder = training_output_folder / "exploratory_estimated"
print("Window policy:", window_policy)
print("Window statuses:", handoff_manifest.get("window_status_counts", {}))
print("Model output folder:", training_output_folder)


## 3. Train and evaluate the model comparisons

Train fresh **external-only**, **local-only** and **combined** forests for the 40-feature and 64-feature designs. All six models use the same independent local test rows. The combined model reuses exactly the same local training rows as the local-only model.

Each training source and the local test set must contain every configured crop. Missing classes, conflicting fields or incomplete features stop fitting with a clear message. An empty pending bundle produces a pending status without fitting a model.

Settings come from the exported configuration snapshot. Freeze the experiment before examining final test results. Each field's confidence budget is shared across its pixels and seasons.

**Outputs:** model bundles, row-level predictions, per-crop precision/recall/F1, macro-F1, balanced accuracy and confusion matrices. The 40-versus-64 comparison is included here. Pixel metrics remain correlated within fields; report independent-field counts alongside them.

**Window checks:** primary training requires reference-supported windows. The separate exploratory policy allows only timing-qualified, reported-label estimates on external training fields. Local test windows must remain reference-supported. An automated pass records algorithmic consistency, not a human review. Frozen settings and local test data are still required; exporting an estimate does not satisfy those requirements.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Training pending: supply reviewed, complete external and independent local rows.` | The loaded table is empty, so fitting is skipped and status is saved as pending_inputs. No accuracy metric has been measured. |
| `No completion message when fitting succeeds` | Successful fitting writes model bundles, predictions, model_settings.json and metrics.json for each feature design. Check the next cell's Training status; this cell does not print an accuracy score. |
| `metrics.json → training_pixels / training_fields: <number>` | Rows and distinct field identities used by each source comparison. The combined model reuses the same local training rows as the local-only comparison. |
| `metrics.json → test_pixels / test_fields: <number>` | Size of the common independent local test set. All comparisons use these same rows; pixels within a field remain correlated. |
| `metrics.json → precision / recall / f1: <value>` | Per-crop performance from 0 to 1. Precision measures how often a predicted crop was correct, recall measures how much of the reference crop was recovered, and F1 balances the two. |
| `metrics.json → macro_f1 / balanced_accuracy: <value>` | Scores from 0 to 1 that give classes equal importance: macro-F1 averages per-class F1; balanced accuracy averages per-class recall. Neither is a pixel-coverage percentage. |
| `metrics.json → confusion_matrix: <counts>` | Rows are reference classes and columns are predicted classes, in the saved classes order. Diagonal entries are correct predictions; off-diagonal entries show confusion. |
<!-- output-guide:end -->


In [ ]:
model_folder = training_output_folder / "models"
model_artifacts = []
comparison_metrics = {}
if partitioned_rows:
    for feature_count, include_indices in [(40, False), (64, True)]:
        destination = model_folder / f"features_{feature_count}"
        comparison_metrics[str(feature_count)] = season_workflow.train_comparisons(
            partitioned_rows, configuration, destination, include_indices=include_indices
        )
        model_artifacts.extend(path for path in destination.iterdir() if path.is_file())
else:
    print("Training pending: supply reviewed, complete external and independent local rows.")
season_workflow.save_json(
    training_output_folder / "training_status.json",
    {
        "status": "completed" if comparison_metrics else "pending_inputs",
        "feature_comparisons": list(comparison_metrics),
    },
)


## 4. Package training results

Save versions, input hashes, configuration, preprocessing coverage and files generated by this training run. The package links results to the exact input bundle without needing to reopen the preprocessing notebook.

**Outputs:** `training_package/run_manifest.json` and `training_package/RUN_README.md`. An input bundle with no prepared rows is recorded as pending. Full-area mapping and real scientific accuracy validation remain separate work.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Training status: completed or pending_inputs` | Completed means the configured model comparisons ran and their outputs were saved. Pending_inputs means no models were fitted from the empty table. Completed does not imply a particular accuracy or validate crop-season evidence. |
| `Training package: <path>` | Folder containing the run manifest, file hashes, saved configuration, preprocessing coverage and result inventory. Inspect its missing_files list before treating the package as complete. |
<!-- output-guide:end -->


In [ ]:
training_coverage = {
    **preprocessing_coverage,
    "sample_counts": season_workflow.sample_counts(partitioned_rows),
    "training_status": "completed" if comparison_metrics else "pending_inputs",
    "training_handoff": str(handoff_path.resolve()),
}
handoff_input_paths = [handoff_path] + [
    handoff_path.parent / record["filename"] for record in handoff_manifest["files"].values()
]
training_manifest = season_workflow.package_run(
    training_output_folder / "training_package",
    configuration,
    input_paths=handoff_input_paths
    + [
        Path("season_workflow.py"),
        Path("crop_workflow.py"),
        Path("timing_quality.py"),
        Path("requirements-notebooks.txt"),
        Path("Crop_Model_Training.ipynb"),
    ],
    artifact_paths=model_artifacts + [training_output_folder / "training_status.json"],
    coverage=training_coverage,
)
print("Training status:", training_coverage["training_status"])
print("Training package:", training_output_folder / "training_package")


## Interpretation and next steps

Inspect `models/features_40/metrics.json` and `models/features_64/metrics.json` together with their field counts and identical test-row predictions. A software test or a synthetic model run does not establish real Benguet classification accuracy.

To change source records, evidence, windows, features or partitions, return to [the preprocessing notebook](Crop_Four_Month_Windows.ipynb) and export a new bundle. The training notebook uses that saved bundle as its input.

See [the review and training guide](docs/training.md) for expected evidence and outputs. Missing thesis components—such as spatial scaling, additional predictors and plot-level uncertainty—remain outside this notebook split.
